# 08_02_サンプラーを_40_行で書く_温度_top-k_

コード：`ch08/blocks/08_02_サンプラーを_40_行で書く_温度_top-k_.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


## 本文の掲載コード（参照）


```python
import torch

NEG = float("-inf")

def apply_repetition_penalty(logits, prev_ids, penalty=1.1):
    """出た語のロジットを縮める。正負で扱いが違う点に注意"""
    if penalty == 1.0 or prev_ids.numel() == 0:
        return logits
    sel = torch.gather(logits, -1, prev_ids)
    # 正なら割り、負なら掛ける（どちらも「小さくする」方向に揃える）
    sel = torch.where(sel > 0, sel / penalty, sel * penalty)
    return logits.scatter(-1, prev_ids, sel)

def sample(logits, *, temperature=1.0, top_k=0, top_p=1.0, min_p=0.0,
           prev_ids=None, repetition_penalty=1.0):
    logits = logits.clone().float()          # ★ fp16 のまま -inf を入れない

    # ① 分布を変形する ------------------------------------------------
    if prev_ids is not None:
        logits = apply_repetition_penalty(logits, prev_ids, repetition_penalty)
    if temperature <= 0:
        return logits.argmax(-1, keepdim=True)   # T=0 は argmax に分岐（0除算回避）。形は (B, 1)
    logits = logits / temperature            # ★ 差が一律 1/T 倍になる

    # ② 台を切り詰める ------------------------------------------------
    if top_k > 0:
        kth = logits.topk(min(top_k, logits.size(-1)), dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, NEG)

    if top_p < 1.0:
        srt, idx = logits.sort(descending=True, dim=-1)
        cum = srt.softmax(-1).cumsum(-1)
        # 累積が p を「超えた本人」は残し、その次から切る
        drop = cum - srt.softmax(-1) >= top_p
        logits = logits.masked_fill(drop.scatter(-1, idx, drop), NEG)

    if min_p > 0.0:
        probs = logits.softmax(-1)
        thresh = min_p * probs.max(-1, keepdim=True).values   # ★ 1位との相対
        logits = logits.masked_fill(probs < thresh, NEG)

    # ③ softmax して抽選 ----------------------------------------------
    probs = logits.softmax(-1)               # -inf → exp(-inf)=0 で自動的に除外
    return torch.multinomial(probs, num_samples=1)
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "08", "--name=sampler", "--skip-docker"], cwd=CODE, check=True)
